# SUMO Environment Setup

In [1]:
"""
Cell 1: Install SUMO and Python dependencies.

SUMO is installed from the Ubuntu package repository. The SUMO_HOME
environment variable is required by sumolib and must be set before any
subsequent imports.
"""

import os
import shutil
import subprocess
import sys

# Terminate any SUMO processes left over from a previous session.
subprocess.run(["pkill", "-9", "-f", "sumo"], capture_output=True)

# Repair any broken package state before attempting installation.
subprocess.run("apt-get clean",               shell=True, capture_output=True)
subprocess.run("rm -rf /var/lib/apt/lists/*", shell=True, capture_output=True)
subprocess.run("apt-get update -qq",          shell=True, capture_output=True)

install = subprocess.run(
    "apt-get install -y sumo sumo-tools",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(
        f"SUMO installation failed:\n{install.stderr[-400:]}"
    )

os.environ["SUMO_HOME"] = "/usr/share/sumo"

sumo_bin       = shutil.which("sumo")
netconvert_bin = shutil.which("netconvert")

if sumo_bin is None:
    raise FileNotFoundError(
        "SUMO binary not found after installation. "
        "Re-run this cell or inspect the apt output above."
    )

subprocess.run(
    "pip install traci torch numpy matplotlib --quiet",
    shell=True,
)

version = subprocess.run(
    [sumo_bin, "--version"], capture_output=True, text=True
)

import torch

print("SUMO path        :", sumo_bin)
print("netconvert path  :", netconvert_bin)
print("SUMO version     :", version.stdout.split("\n")[0])
print("PyTorch version  :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
print("SUMO_HOME        :", os.environ["SUMO_HOME"])

SUMO path        : /usr/bin/sumo
netconvert path  : /usr/bin/netconvert
SUMO version     : Eclipse SUMO sumo Version 1.12.0
PyTorch version  : 2.11.0+cpu
CUDA available   : False
SUMO_HOME        : /usr/share/sumo


In [2]:
"""
Cell 2: Generate SUMO node and edge XML files and compile them into a
road network using netconvert.

Two lanes per edge are required so that all 12 turning movements can be
accommodated without lane conflicts at the central junction.

Note: netconvert exits with return code -11 (SIGSEGV) on some Colab
builds after successfully writing the output file. The assertion below
checks file size rather than return code to handle this known issue.
"""

import os
import subprocess

os.makedirs("sumo_config", exist_ok=True)

NODES_XML = """<?xml version="1.0" encoding="UTF-8"?>
<nodes>
    <node id="center" x="0"    y="0"    type="traffic_light"/>
    <node id="north"  x="0"    y="500"  type="priority"/>
    <node id="south"  x="0"    y="-500" type="priority"/>
    <node id="east"   x="500"  y="0"    type="priority"/>
    <node id="west"   x="-500" y="0"    type="priority"/>
</nodes>"""

EDGES_XML = """<?xml version="1.0" encoding="UTF-8"?>
<edges>
    <edge id="N2C" from="north"  to="center" numLanes="2" speed="13.89"/>
    <edge id="S2C" from="south"  to="center" numLanes="2" speed="13.89"/>
    <edge id="E2C" from="east"   to="center" numLanes="2" speed="13.89"/>
    <edge id="W2C" from="west"   to="center" numLanes="2" speed="13.89"/>
    <edge id="C2N" from="center" to="north"  numLanes="2" speed="13.89"/>
    <edge id="C2S" from="center" to="south"  numLanes="2" speed="13.89"/>
    <edge id="C2E" from="center" to="east"   numLanes="2" speed="13.89"/>
    <edge id="C2W" from="center" to="west"   numLanes="2" speed="13.89"/>
</edges>"""

with open("sumo_config/intersection.nod.xml", "w") as f:
    f.write(NODES_XML)
with open("sumo_config/intersection.edg.xml", "w") as f:
    f.write(EDGES_XML)

result = subprocess.run(
    [
        "netconvert",
        "--node-files=sumo_config/intersection.nod.xml",
        "--edge-files=sumo_config/intersection.edg.xml",
        "--output-file=sumo_config/intersection.net.xml",
        "--tls.guess=true",
        "--tls.guess-signals=true",
        "--no-warnings",
    ],
    capture_output=True,
    text=True,
)

net_path = "sumo_config/intersection.net.xml"
net_size = os.path.getsize(net_path) if os.path.exists(net_path) else 0

if net_size < 5000:
    raise RuntimeError(
        f"Network file is too small ({net_size} bytes). "
        f"netconvert stderr:\n{result.stderr[:400]}"
    )

print(f"netconvert return code : {result.returncode}"
      f"  (-11 = known Colab segfault after successful write, safe to ignore)")
print(f"Network file size      : {net_size:,} bytes")
print("Road network built successfully.")

netconvert return code : -11  (-11 = known Colab segfault after successful write, safe to ignore)
Network file size      : 20,465 bytes
Road network built successfully.


In [3]:
"""
Cell 3: Read the compiled network and derive the correct traffic-light
phase strings automatically.

The length of each phase string equals the number of controlled
connections, which netconvert determines from the network topology.
Building the strings programmatically avoids hard-coding a length that
may vary across SUMO versions or build configurations.
"""

import os
import sys

sys.path.append(os.path.join(os.environ["SUMO_HOME"], "tools"))
import sumolib


def _build_phase_strings(green_indices: list, length: int) -> tuple:
    """
    Construct a (green_string, yellow_string) pair of the given length.

    Positions in green_indices are set to 'G' and 'y' respectively;
    all other positions remain 'r'.
    """
    green  = ["r"] * length
    yellow = ["r"] * length
    for i in green_indices:
        green[i]  = "G"
        yellow[i] = "y"
    return "".join(green), "".join(yellow)


net   = sumolib.net.readNet("sumo_config/intersection.net.xml")
tls   = net.getTrafficLights()[0]
conns = tls.getConnections()
n     = len(conns)

ns_indices: list = []
ew_indices: list = []

for idx, (from_lane, _to_lane, _link) in enumerate(conns):
    edge_id = from_lane.getEdge().getID()
    if edge_id in ("N2C", "S2C"):
        ns_indices.append(idx)
    elif edge_id in ("E2C", "W2C"):
        ew_indices.append(idx)

NS_GREEN,  NS_YELLOW = _build_phase_strings(ns_indices, n)
EW_GREEN,  EW_YELLOW = _build_phase_strings(ew_indices, n)

print(f"Traffic light ID           : {tls.getID()}")
print(f"Controlled connections     : {n}  (phase string length = {n})")
print(f"NS connection indices ({len(ns_indices)})  : {ns_indices}")
print(f"EW connection indices ({len(ew_indices)})  : {ew_indices}")
print()
print(f"NS_GREEN  = '{NS_GREEN}'")
print(f"NS_YELLOW = '{NS_YELLOW}'")
print(f"EW_GREEN  = '{EW_GREEN}'")
print(f"EW_YELLOW = '{EW_YELLOW}'")

Traffic light ID           : center
Controlled connections     : 20  (phase string length = 20)
NS connection indices (10)  : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
EW connection indices (10)  : [0, 1, 2, 3, 4, 15, 16, 17, 18, 19]

NS_GREEN  = 'rrrrrGGGGGGGGGGrrrrr'
NS_YELLOW = 'rrrrryyyyyyyyyyrrrrr'
EW_GREEN  = 'GGGGGrrrrrrrrrrGGGGG'
EW_YELLOW = 'yyyyyrrrrrrrrrryyyyy'


In [4]:
"""
Cell 4: Convert observed Bangalore vehicle counts into SUMO flow
definitions and write the route and configuration files.

Source data (total vehicles passing the intersection per period):
    00:00-08:00    550 vehicles   (9 h, night)
    08:00-11:00  4,000 vehicles   (3 h, morning rush)
    11:00-13:00  2,850 vehicles   (2 h, mid-morning)
    13:00-16:00  2,100 vehicles   (3 h, afternoon)
    16:00-20:00  4,400 vehicles   (4 h, evening rush)
    20:00-23:00  2,500 vehicles   (3 h, evening)
    23:00-24:00    550 vehicles   (1 h, night tail)
    Total        16,400 vehicles / day

Conversion formula:
    veh/hr per route = total_veh / period_hours / 12 routes

Vehicle mix (typical Bangalore urban intersection):
    60 % cars, 30 % motorcycles, 10 % auto-rickshaws
"""

import os

SECONDS_PER_HOUR = 3600

# Schedule split across midnight so no period crosses the 86400 s boundary.
# Each entry: (begin_s, end_s, total_vehicles, label)
BANGALORE_SCHEDULE = [
    (0,                       8  * SECONDS_PER_HOUR,   550, "Night (00:00-08:00)"),
    (8  * SECONDS_PER_HOUR,   11 * SECONDS_PER_HOUR,  4000, "Morning rush (08:00-11:00)"),
    (11 * SECONDS_PER_HOUR,   13 * SECONDS_PER_HOUR,  2850, "Mid-morning (11:00-13:00)"),
    (13 * SECONDS_PER_HOUR,   16 * SECONDS_PER_HOUR,  2100, "Afternoon (13:00-16:00)"),
    (16 * SECONDS_PER_HOUR,   20 * SECONDS_PER_HOUR,  4400, "Evening rush (16:00-20:00)"),
    (20 * SECONDS_PER_HOUR,   23 * SECONDS_PER_HOUR,  2500, "Evening (20:00-23:00)"),
    (23 * SECONDS_PER_HOUR,   86400,                    550, "Night (23:00-24:00)"),
]

# All 12 turning movements: straight, left, and right from each arm.
ALL_ROUTES = [
    ("route_ns", "N2C C2S"),   # North -> South  (straight)
    ("route_nw", "N2C C2W"),   # North -> West   (left turn)
    ("route_ne", "N2C C2E"),   # North -> East   (right turn)
    ("route_sn", "S2C C2N"),   # South -> North  (straight)
    ("route_sw", "S2C C2W"),   # South -> West   (right turn)
    ("route_se", "S2C C2E"),   # South -> East   (left turn)
    ("route_en", "E2C C2N"),   # East  -> North  (left turn)
    ("route_ew", "E2C C2W"),   # East  -> West   (straight)
    ("route_es", "E2C C2S"),   # East  -> South  (right turn)
    ("route_wn", "W2C C2N"),   # West  -> North  (right turn)
    ("route_we", "W2C C2E"),   # West  -> East   (straight)
    ("route_ws", "W2C C2S"),   # West  -> South  (left turn)
]

VEHICLE_TYPES_XML = (
    '    <vType id="car"  accel="2.6" decel="4.5" sigma="0.5" '
    'length="5" minGap="2.5" maxSpeed="13.89"/>\n'
    '    <vType id="bike" accel="3.0" decel="5.0" sigma="0.6" '
    'length="2" minGap="1.5" maxSpeed="11.11" vClass="motorcycle"/>\n'
    '    <vType id="auto" accel="2.0" decel="4.0" sigma="0.5" '
    'length="4" minGap="2.0" maxSpeed="11.11"/>'
)

TRAFFIC_MIX = {"car": 0.60, "bike": 0.30, "auto": 0.10}


def make_route_file(filepath: str) -> None:
    """Write the SUMO route file containing vehicle types and flow definitions."""
    route_tags = "\n".join(
        f'    <route id="{rid}" edges="{edges}"/>'
        for rid, edges in ALL_ROUTES
    )

    flow_tags = []
    flow_id   = 0

    print(f"\n{'Period':<30} {'Total':>7} {'Hours':>6} "
          f"{'Total/hr':>10} {'Per route':>10}")
    print("-" * 68)

    for begin, end, total_veh, label in BANGALORE_SCHEDULE:
        duration_hr  = (end - begin) / SECONDS_PER_HOUR
        total_per_hr = total_veh / duration_hr
        per_route_hr = total_per_hr / len(ALL_ROUTES)

        print(f"{label:<30} {total_veh:>7} {duration_hr:>6.1f} "
              f"{total_per_hr:>10.0f} {per_route_hr:>10.1f}")

        for route_id, _ in ALL_ROUTES:
            tag = route_id.removeprefix("route_")
            for vtype, proportion in TRAFFIC_MIX.items():
                rate = per_route_hr * proportion
                flow_tags.append(
                    f'    <flow id="f{flow_id:04d}_{vtype}_{tag}" '
                    f'route="{route_id}" type="{vtype}" '
                    f'begin="{begin}" end="{end}" '
                    f'vehsPerHour="{rate:.1f}" '
                    f'departSpeed="max" departPos="base" departLane="best"/>'
                )
                flow_id += 1

    xml = (
        '<?xml version="1.0" encoding="UTF-8"?>\n<routes>\n'
        + VEHICLE_TYPES_XML + "\n\n"
        + route_tags + "\n\n"
        + "\n".join(flow_tags) + "\n</routes>"
    )
    with open(filepath, "w") as f:
        f.write(xml)

    print(f"\nRoute file written : {filepath}  ({flow_id} flow entries)")


def make_sumocfg(route_filename: str, cfg_path: str,
                 duration: int = 86400, step_length: int = 10) -> None:
    """Write the SUMO configuration file for a single simulation run."""
    xml = f"""<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <input>
        <net-file value="intersection.net.xml"/>
        <route-files value="{route_filename}"/>
    </input>
    <time>
        <begin value="0"/>
        <end value="{duration}"/>
        <step-length value="{step_length}"/>
    </time>
</configuration>"""
    with open(cfg_path, "w") as f:
        f.write(xml)


make_route_file("sumo_config/routes_bangalore.rou.xml")
make_sumocfg(
    "routes_bangalore.rou.xml",
    "sumo_config/sim_bangalore.sumocfg",
    duration=86400,
    step_length=10,
)

print("\nSimulation parameters:")
print(f"  Duration      : 86,400 s  (24 h)")
print(f"  Step length   : 10 s  ->  8,640 decision steps per episode")
print(f"  Routes        : {len(ALL_ROUTES)}  (all turning movements)")
print(f"  Vehicle mix   : 60% car / 30% motorcycle / 10% auto-rickshaw")
print(f"  Total / day   : 16,400 vehicles")


Period                           Total  Hours   Total/hr  Per route
--------------------------------------------------------------------
Night (00:00-08:00)                550    8.0         69        5.7
Morning rush (08:00-11:00)        4000    3.0       1333      111.1
Mid-morning (11:00-13:00)         2850    2.0       1425      118.8
Afternoon (13:00-16:00)           2100    3.0        700       58.3
Evening rush (16:00-20:00)        4400    4.0       1100       91.7
Evening (20:00-23:00)             2500    3.0        833       69.4
Night (23:00-24:00)                550    1.0        550       45.8

Route file written : sumo_config/routes_bangalore.rou.xml  (252 flow entries)

Simulation parameters:
  Duration      : 86,400 s  (24 h)
  Step length   : 10 s  ->  8,640 decision steps per episode
  Routes        : 12  (all turning movements)
  Vehicle mix   : 60% car / 30% motorcycle / 10% auto-rickshaw
  Total / day   : 16,400 vehicles


In [5]:
"""
Cell 5: Run a short headless SUMO simulation to confirm that the network,
route, and configuration files are internally consistent before training
begins.
"""

import subprocess

result = subprocess.run(
    [
        "sumo",
        "-c", "sumo_config/sim_bangalore.sumocfg",
        "--no-step-log",
        "--no-warnings",
        "--end", "500",
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("STDOUT:", result.stdout[:300])
    print("STDERR:", result.stderr[:400])
    raise RuntimeError(
        f"SUMO validation failed (return code {result.returncode})."
    )

print(f"Return code : {result.returncode}")
print(f"SUMO output : {result.stdout.strip() or '(none)'}")
print("Configuration is valid. Ready to train.")

Return code : 0
SUMO output : (none)
Configuration is valid. Ready to train.


# RL Agent Initialization

In [6]:
"""
Cell 6: Define the shared-backbone Actor-Critic neural network.

Architecture:
    - Two-layer backbone with 64 hidden units and Tanh activation.
    - Separate linear heads for the actor (action logits) and
      critic (state value estimate).
    - Orthogonal weight initialisation, which is standard practice
      for PPO and improves early training stability.

State dimension  : 8 (normalised queue lengths, waiting times, speeds,
                      current phase, and time-of-day fraction)
Action space     : 2 discrete actions (NS-green = 0, EW-green = 1)
"""

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device : {DEVICE}")


class ActorCritic(nn.Module):
    """
    Shared-backbone actor-critic network for discrete action spaces.

    Parameters
    ----------
    state_dim  : Dimension of the input state vector.
    action_dim : Number of discrete actions.
    hidden     : Width of each hidden layer.
    """

    def __init__(self, state_dim: int = 8,
                 action_dim: int = 2,
                 hidden: int = 64) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, action_dim)
        self.critic = nn.Linear(hidden, 1)
        self._init_weights()

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.zeros_(module.bias)
        nn.init.orthogonal_(self.actor.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x: torch.Tensor):
        features = self.backbone(x)
        return self.actor(features), self.critic(features)

    def get_action(self, state: np.ndarray):
        """Sample one action from the current policy without gradient tracking."""
        with torch.no_grad():
            tensor        = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            logits, value = self(tensor)
            dist          = Categorical(logits=logits)
            action        = dist.sample()
        return action.item(), dist.log_prob(action), value

    def evaluate(self, states: torch.Tensor, actions: torch.Tensor):
        """Evaluate a batch of state-action pairs for the PPO update step."""
        logits, values = self(states)
        dist           = Categorical(logits=logits)
        log_probs      = dist.log_prob(actions)
        entropy        = dist.entropy()
        return log_probs, values.squeeze(-1), entropy


n_params = sum(p.numel() for p in ActorCritic().parameters())
print(f"Network parameters : {n_params:,}")
print(f"Hidden width       : 64")
print(f"Activation         : Tanh")
print(f"Weight init        : Orthogonal")

Compute device : cpu
Network parameters : 4,931
Hidden width       : 64
Activation         : Tanh
Weight init        : Orthogonal


In [7]:
"""
Cell 7: Define the PPO agent and the Fixed-Cycle baseline policy.

PPO hyperparameters (matched to the 2WSI-RL benchmark where applicable):
    gamma        = 0.95   discount factor
    lr           = 3e-4   Adam learning rate (equivalent role to alpha)
    eps_clip     = 0.2    clipping range (analogous to epsilon)
    gae_lambda   = 0.95   generalised advantage estimation lambda
    k_epochs     = 8      update passes per collected rollout
    rollout_len  = 1024   steps collected before each update

Reward normalisation:
    The raw reward (negative sum of edge waiting times) can reach -50,000
    per step under heavy traffic. It is divided by 1,000 and clipped to
    [-10, 0] to keep gradient magnitudes stable across traffic periods.
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


class PPOAgent:
    """
    Proximal Policy Optimisation agent with clipped surrogate objective,
    Generalised Advantage Estimation (GAE), and mini-batch updates.
    """

    def __init__(
        self,
        state_dim:    int   = 8,
        action_dim:   int   = 2,
        lr:           float = 3e-4,
        gamma:        float = 0.95,
        eps_clip:     float = 0.2,
        gae_lambda:   float = 0.95,
        k_epochs:     int   = 8,
        rollout_len:  int   = 1024,
        entropy_coef: float = 0.05,
        vf_coef:      float = 0.5,
        mini_batch:   int   = 256,
    ) -> None:
        self.gamma        = gamma
        self.eps_clip     = eps_clip
        self.gae_lambda   = gae_lambda
        self.k_epochs     = k_epochs
        self.rollout_len  = rollout_len
        self.entropy_coef = entropy_coef
        self.vf_coef      = vf_coef
        self.mini_batch   = mini_batch

        self.net     = ActorCritic(state_dim, action_dim).to(DEVICE)
        self.opt     = optim.Adam(self.net.parameters(), lr=lr, eps=1e-5)
        self.lr_init = lr
        self.lr_now  = lr

        self._clear_buffer()
        self.total_steps:  int  = 0
        self.update_count: int  = 0
        self.policy_losses: list = []
        self.value_losses:  list = []
        self.entropy_hist:  list = []

    # ------------------------------------------------------------------
    # Rollout buffer
    # ------------------------------------------------------------------

    def _clear_buffer(self) -> None:
        self.buf_states:   list = []
        self.buf_actions:  list = []
        self.buf_logprobs: list = []
        self.buf_rewards:  list = []
        self.buf_dones:    list = []
        self.buf_values:   list = []

    # ------------------------------------------------------------------
    # Interaction
    # ------------------------------------------------------------------

    def choose_action(self, state: np.ndarray) -> int:
        """Choose an action and store the transition in the rollout buffer."""
        action, log_prob, value = self.net.get_action(state)
        self.buf_states.append(state)
        self.buf_actions.append(action)
        self.buf_logprobs.append(log_prob.item())
        self.buf_values.append(value.item())
        self.total_steps += 1
        return action

    def store_reward(self, reward: float, done: bool) -> None:
        """Normalise and store the reward for the last action taken."""
        normalised = max(reward / 1000.0, -10.0)
        self.buf_rewards.append(normalised)
        self.buf_dones.append(float(done))

    # ------------------------------------------------------------------
    # Learning
    # ------------------------------------------------------------------

    def update(self) -> None:
        """Run a PPO update step when the rollout buffer is full."""
        if len(self.buf_rewards) < self.rollout_len:
            return

        rewards = np.array(self.buf_rewards, dtype=np.float32)
        dones   = np.array(self.buf_dones,   dtype=np.float32)
        values  = np.array(self.buf_values,  dtype=np.float32)

        # Generalised Advantage Estimation
        advantages = np.zeros_like(rewards)
        last_adv   = 0.0
        for t in reversed(range(len(rewards))):
            next_val      = values[t + 1] if t + 1 < len(values) else 0.0
            delta         = (
                rewards[t]
                + self.gamma * next_val * (1.0 - dones[t])
                - values[t]
            )
            advantages[t] = last_adv = (
                delta
                + self.gamma * self.gae_lambda * (1.0 - dones[t]) * last_adv
            )

        returns    = advantages + values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        t_states  = torch.FloatTensor(np.array(self.buf_states)).to(DEVICE)
        t_actions = torch.LongTensor(self.buf_actions).to(DEVICE)
        t_old_lp  = torch.FloatTensor(self.buf_logprobs).to(DEVICE)
        t_adv     = torch.FloatTensor(advantages).to(DEVICE)
        t_ret     = torch.FloatTensor(returns).to(DEVICE)

        p_losses, v_losses, entropies = [], [], []

        for _ in range(self.k_epochs):
            indices = np.random.permutation(len(t_states))
            for start in range(0, len(indices), self.mini_batch):
                mb = indices[start : start + self.mini_batch]

                new_lp, val_pred, entropy = self.net.evaluate(
                    t_states[mb], t_actions[mb]
                )
                ratio = torch.exp(new_lp - t_old_lp[mb])
                surr1 = ratio * t_adv[mb]
                surr2 = torch.clamp(
                    ratio, 1.0 - self.eps_clip, 1.0 + self.eps_clip
                ) * t_adv[mb]

                p_loss = -torch.min(surr1, surr2).mean()
                v_loss = nn.MSELoss()(val_pred, t_ret[mb])
                e_loss = -entropy.mean()
                loss   = p_loss + self.vf_coef * v_loss + self.entropy_coef * e_loss

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
                self.opt.step()

                p_losses.append(p_loss.item())
                v_losses.append(v_loss.item())
                entropies.append(entropy.mean().item())

        self.policy_losses.append(float(np.mean(p_losses)))
        self.value_losses.append(float(np.mean(v_losses)))
        self.entropy_hist.append(float(np.mean(entropies)))
        self.update_count += 1
        self._clear_buffer()

    def lr_decay(self, episode: int, total_episodes: int) -> None:
        """Linearly anneal the learning rate from lr_init to lr_init / 10."""
        frac = 1.0 - 0.9 * (episode / total_episodes)
        self.lr_now = self.lr_init * frac
        for group in self.opt.param_groups:
            group["lr"] = self.lr_now

    def stats(self) -> dict:
        """Return a summary of the current training state."""
        return {
            "updates"  : self.update_count,
            "steps"    : self.total_steps,
            "p_loss"   : round(self.policy_losses[-1], 4) if self.policy_losses else 0.0,
            "v_loss"   : round(self.value_losses[-1],  4) if self.value_losses  else 0.0,
            "entropy"  : round(self.entropy_hist[-1],  4) if self.entropy_hist  else 0.0,
            "lr"       : round(self.lr_now, 6),
        }


print("PPO agent defined.")
print(f"  gamma        : 0.95")
print(f"  lr           : 3e-4  (annealed to 3e-5 over training)")
print(f"  eps_clip     : 0.2")
print(f"  gae_lambda   : 0.95")
print(f"  k_epochs     : 8")
print(f"  rollout_len  : 1024")
print(f"  mini_batch   : 256")

PPO agent defined.
  gamma        : 0.95
  lr           : 3e-4  (annealed to 3e-5 over training)
  eps_clip     : 0.2
  gae_lambda   : 0.95
  k_epochs     : 8
  rollout_len  : 1024
  mini_batch   : 256


In [8]:
"""
Cell 8: Final SUMO environment definition used for all training and
evaluation runs.

Key design decisions:
    - Metrics are collected only for vehicles currently on incoming edges.
      Measuring all active vehicles inflates the mean waiting time with
      vehicles that are already through the intersection and moving freely.
    - Delay is computed against the free-flow travel time for a 500 m edge
      (approximately 36 s at maximum speed).
    - Travel time is tracked from a vehicle's first appearance in the
      simulation to the step it is recorded as arrived.

State vector (8-dimensional, all values normalised to [0, 1]):
    [0] NS halting vehicle count
    [1] EW halting vehicle count
    [2] NS total waiting time
    [3] EW total waiting time
    [4] NS mean speed
    [5] EW mean speed
    [6] Current phase  (0 = NS-green, 1 = EW-green)
    [7] Time of day    (fraction of total episode length)

Action space : 2  (0 = NS-green, 1 = EW-green)
Reward       : Negative sum of waiting times on all incoming edges per step.
"""

import os
import socket
import subprocess
import time
import xml.etree.ElementTree as ET

import numpy as np
import traci

STEP_LENGTH = 10  # simulated seconds per simulation step

# Free-flow travel time per vehicle type for a 500 m edge.
FREE_FLOW_TIME = {
    "car" : 500.0 / 13.89,   # approx. 36.0 s
    "bike": 500.0 / 11.11,   # approx. 45.0 s
    "auto": 500.0 / 11.11,   # approx. 45.0 s
}
DEFAULT_FREE_FLOW = 500.0 / 13.89


def _get_free_port() -> int:
    """Ask the OS to allocate an available ephemeral TCP port."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        return s.getsockname()[1]


class BangaloreSumoEnv:
    """
    SUMO gym-style environment for a single signalised intersection
    loaded with Bangalore 24-hour demand data.

    Parameters
    ----------
    cfg_path : Path to the .sumocfg file.
    """

    PHASES = {
        0: (NS_GREEN,  NS_YELLOW),
        1: (EW_GREEN,  EW_YELLOW),
    }

    # Phase timing in simulation steps (1 step = STEP_LENGTH seconds)
    YELLOW_DURATION = 1   #  1 step  = 10 s
    MIN_GREEN       = 8   #  8 steps = 80 s

    # Normalisation bounds calibrated to Bangalore peak-hour conditions
    MAX_QUEUE = 80.0
    MAX_WAIT  = 600.0
    MAX_SPEED = 13.89

    # Metric collection scope: incoming edges only
    INCOMING_EDGES = ("N2C", "S2C", "E2C", "W2C")

    def __init__(self, cfg_path: str) -> None:
        self.cfg_path      = os.path.abspath(cfg_path)
        self.tls_id        = "center"
        self.current_step  = 0
        self.current_phase = 0
        self._running      = False
        self._proc         = None
        self.max_steps     = self._parse_max_steps()
        self._reset_metric_buffers()

    # ------------------------------------------------------------------
    # Configuration helpers
    # ------------------------------------------------------------------

    def _parse_max_steps(self) -> int:
        tree     = ET.parse(self.cfg_path)
        end_tag  = tree.find(".//end")
        step_tag = tree.find(".//step-length")
        end_val  = int(end_tag.get("value",  86400)) if end_tag  is not None else 86400
        step_val = int(step_tag.get("value", 10))    if step_tag is not None else 10
        return end_val // step_val

    def _reset_metric_buffers(self) -> None:
        # Benchmark metrics (matching 2WSI-RL evaluation protocol)
        self._buf_total_stopped:      list = []
        self._buf_total_waiting_time: list = []
        self._buf_mean_waiting_time:  list = []
        self._buf_mean_speed:         list = []
        # Additional metrics
        self._buf_queue_length:       list = []
        self._buf_throughput:         list = []
        self._buf_avg_travel_time:    list = []
        self._buf_delay:              list = []
        # Internal travel-time tracking
        self._vehicles_arrived:   int  = 0
        self._total_travel_time:  float = 0.0
        self._vehicle_entry_step: dict = {}

    # ------------------------------------------------------------------
    # Core interface
    # ------------------------------------------------------------------

    def reset(self) -> np.ndarray:
        """Start a new SUMO episode and return the initial observation."""
        self._safe_close()
        self._reset_metric_buffers()

        port = _get_free_port()
        cmd  = [
            "sumo",
            "-c", self.cfg_path,
            "--no-step-log",
            "--no-warnings",
            "--random",
            "--quit-on-end",
            "--remote-port", str(port),
        ]
        self._proc = subprocess.Popen(
            cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
        )
        time.sleep(1.0)

        if self._proc.poll() is not None:
            stderr = self._proc.stderr.read().decode()
            raise RuntimeError(f"SUMO crashed on startup:\n{stderr}")

        traci.init(port=port, numRetries=10)
        self._running      = True
        self.current_step  = 0
        self.current_phase = 0
        traci.trafficlight.setRedYellowGreenState(
            self.tls_id, self.PHASES[0][0]
        )
        return self._observe()

    def step(self, action: int):
        """
        Apply the chosen action, advance the simulation, and record metrics.

        Returns
        -------
        observation : np.ndarray  Next state vector.
        reward      : float       Negative total waiting time this step.
        done        : bool        True when the episode has ended.
        """
        if action != self.current_phase:
            traci.trafficlight.setRedYellowGreenState(
                self.tls_id, self.PHASES[self.current_phase][1]
            )
            for _ in range(self.YELLOW_DURATION):
                traci.simulationStep()
                self._record()
                self.current_step += 1

        traci.trafficlight.setRedYellowGreenState(
            self.tls_id, self.PHASES[action][0]
        )
        self.current_phase = action

        reward = 0.0
        for _ in range(self.MIN_GREEN):
            traci.simulationStep()
            self._record()
            self.current_step += 1
            reward -= sum(
                traci.edge.getWaitingTime(e) for e in self.INCOMING_EDGES
            )

        done = self.current_step >= self.max_steps
        return self._observe(), reward, done

    def close(self) -> None:
        """Terminate the SUMO process and release all resources."""
        self._safe_close()

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _observe(self) -> np.ndarray:
        """Construct the normalised 8-dimensional state vector."""
        ns_queue = sum(traci.edge.getLastStepHaltingNumber(e)
                       for e in ("N2C", "S2C"))
        ew_queue = sum(traci.edge.getLastStepHaltingNumber(e)
                       for e in ("E2C", "W2C"))
        ns_wait  = sum(traci.edge.getWaitingTime(e) for e in ("N2C", "S2C"))
        ew_wait  = sum(traci.edge.getWaitingTime(e) for e in ("E2C", "W2C"))
        ns_speed = float(np.mean([
            traci.edge.getLastStepMeanSpeed(e) for e in ("N2C", "S2C")
        ]))
        ew_speed = float(np.mean([
            traci.edge.getLastStepMeanSpeed(e) for e in ("E2C", "W2C")
        ]))
        time_of_day = self.current_step / max(self.max_steps, 1)

        return np.array([
            min(ns_queue / self.MAX_QUEUE,  1.0),
            min(ew_queue / self.MAX_QUEUE,  1.0),
            min(ns_wait  / self.MAX_WAIT,   1.0),
            min(ew_wait  / self.MAX_WAIT,   1.0),
            min(ns_speed / self.MAX_SPEED,  1.0),
            min(ew_speed / self.MAX_SPEED,  1.0),
            float(self.current_phase),
            float(time_of_day),
        ], dtype=np.float32)

    def _record(self) -> None:
        """Record all metrics for the current simulation step."""
        # Restrict measurement to vehicles on incoming edges only.
        incoming_vids: list = []
        for edge in self.INCOMING_EDGES:
            incoming_vids.extend(traci.edge.getLastStepVehicleIDs(edge))

        # Track all vehicles for network-wide travel time.
        for vid in traci.vehicle.getIDList():
            if vid not in self._vehicle_entry_step:
                self._vehicle_entry_step[vid] = self.current_step

        arrived = traci.simulation.getArrivedIDList()
        for vid in arrived:
            if vid in self._vehicle_entry_step:
                travel = (
                    (self.current_step - self._vehicle_entry_step[vid])
                    * STEP_LENGTH
                )
                self._total_travel_time += travel
                self._vehicles_arrived  += 1
        self._buf_throughput.append(len(arrived))

        avg_travel = self._total_travel_time / max(self._vehicles_arrived, 1)
        n          = len(incoming_vids)

        if n == 0:
            self._buf_total_stopped.append(0)
            self._buf_total_waiting_time.append(0.0)
            self._buf_mean_waiting_time.append(0.0)
            self._buf_mean_speed.append(0.0)
            self._buf_queue_length.append(0)
            self._buf_avg_travel_time.append(avg_travel)
            self._buf_delay.append(0.0)
            return

        speeds = [traci.vehicle.getSpeed(v)       for v in incoming_vids]
        waits  = [traci.vehicle.getWaitingTime(v)  for v in incoming_vids]

        # Per-vehicle delay relative to type-specific free-flow travel time.
        delays = []
        for vid in incoming_vids:
            try:
                ff = FREE_FLOW_TIME.get(traci.vehicle.getTypeID(vid),
                                        DEFAULT_FREE_FLOW)
                delays.append(max(0.0, traci.vehicle.getWaitingTime(vid) - ff))
            except Exception:
                delays.append(0.0)

        total_queue = sum(
            traci.edge.getLastStepHaltingNumber(e)
            for e in self.INCOMING_EDGES
        )

        self._buf_total_stopped.append(int(sum(1 for s in speeds if s < 0.1)))
        self._buf_total_waiting_time.append(float(sum(waits)))
        self._buf_mean_waiting_time.append(float(np.mean(waits)))
        self._buf_mean_speed.append(float(np.mean(speeds)))
        self._buf_queue_length.append(int(total_queue))
        self._buf_avg_travel_time.append(avg_travel)
        self._buf_delay.append(float(np.mean(delays)) if delays else 0.0)

    def _safe_close(self) -> None:
        if self._running:
            try:
                traci.close()
            except Exception:
                pass
            self._running = False
        if self._proc is not None:
            try:
                self._proc.kill()
                self._proc.wait()
            except Exception:
                pass
            self._proc = None
        subprocess.run(["pkill", "-9", "-f", "sumo"], capture_output=True)
        time.sleep(0.5)

    def get_metrics(self) -> dict:
        """Return all per-step metric arrays for the completed episode."""
        return {
            # Benchmark metrics (2WSI-RL evaluation protocol)
            "system_total_stopped"     : np.array(self._buf_total_stopped),
            "system_total_waiting_time": np.array(self._buf_total_waiting_time),
            "system_mean_waiting_time" : np.array(self._buf_mean_waiting_time),
            "system_mean_speed"        : np.array(self._buf_mean_speed),
            # Additional metrics
            "avg_waiting_time"         : np.array(self._buf_mean_waiting_time),
            "avg_travel_time"          : np.array(self._buf_avg_travel_time),
            "queue_length"             : np.array(self._buf_queue_length),
            "throughput"               : np.array(self._buf_throughput),
            "delay"                    : np.array(self._buf_delay),
        }

    @property
    def n_actions(self) -> int:
        return 2

    @property
    def state_dim(self) -> int:
        return 8


print("BangaloreSumoEnv defined.")
print(f"  Step length      : {STEP_LENGTH} s")
print(f"  Yellow duration  : {BangaloreSumoEnv.YELLOW_DURATION} step  "
      f"= {BangaloreSumoEnv.YELLOW_DURATION * STEP_LENGTH} s")
print(f"  Minimum green    : {BangaloreSumoEnv.MIN_GREEN} steps "
      f"= {BangaloreSumoEnv.MIN_GREEN * STEP_LENGTH} s")
print(f"  Steps per episode: {86400 // STEP_LENGTH:,}")
print(f"  Metrics          : 9  (4 benchmark + 5 additional)")
print(f"  Measurement scope: incoming edges only")

BangaloreSumoEnv defined.
  Step length      : 10 s
  Yellow duration  : 1 step  = 10 s
  Minimum green    : 8 steps = 80 s
  Steps per episode: 8,640
  Metrics          : 9  (4 benchmark + 5 additional)
  Measurement scope: incoming edges only


# Fixed-Policy Initialization

In [9]:
"""
Cell 9: Fixed-cycle baseline policy used as a performance reference.

The policy alternates between NS-green and EW-green on a fixed schedule,
mimicking the fixedcycle agent from the 2WSI-RL benchmark repository.

With step_length = 10 s and cycle_steps = 3:
    Each phase lasts  3 decisions x 10 s = 30 s of simulated time.
    A full cycle is   NS green (30 s) + EW green (30 s) = 60 s.
"""


class FixedCyclePolicy:
    """
    Deterministic fixed-cycle traffic signal controller.

    Parameters
    ----------
    cycle_steps : Number of decisions per phase before switching.
                  With step_length=10 s, cycle_steps=3 gives 30 s per phase.
    """

    def __init__(self, cycle_steps: int = 3) -> None:
        self.cycle_steps  = cycle_steps
        self.step_counter = 0

    def choose_action(self, state) -> int:
        """Return 0 (NS-green) or 1 (EW-green) based on the current cycle position."""
        phase = (self.step_counter // self.cycle_steps) % 2
        self.step_counter += 1
        return int(phase)

    def reset(self) -> None:
        """Reset the cycle counter for a new evaluation episode."""
        self.step_counter = 0


FIXED_POLICY = FixedCyclePolicy(cycle_steps=3)

print("FixedCyclePolicy defined.")
print(f"  cycle_steps : 3  ->  30 s per phase  (NS-green then EW-green)")
print(f"  Full cycle  : 60 s")

FixedCyclePolicy defined.
  cycle_steps : 3  ->  30 s per phase  (NS-green then EW-green)
  Full cycle  : 60 s


# Phase 1 Training
100 episodes

In [10]:
"""
Cell 10: Phase 1 training — run the PPO agent for 100 episodes on the
Bangalore 24-hour traffic schedule.

The best model weights are saved to disk after every episode that achieves
a new highest cumulative reward. Learning rate is linearly annealed from
3e-4 to 3e-5 over the 100 episodes.
"""

import time

import numpy as np
import torch

N_EPISODES  = 100
CHECKPOINT  = "best_ppo_bangalore.pt"

agent = PPOAgent(
    state_dim    = 8,
    action_dim   = 2,
    lr           = 3e-4,
    gamma        = 0.95,
    eps_clip     = 0.2,
    gae_lambda   = 0.95,
    k_epochs     = 8,
    rollout_len  = 1024,
    entropy_coef = 0.05,
    vf_coef      = 0.5,
)

train_rewards:      list  = []
train_metrics_last        = None
best_reward:        float = float("-inf")

print(f"Phase 1 training — {N_EPISODES} episodes")
print(f"{'Ep':>4} | {'Reward':>14} | {'Updates':>7} | "
      f"{'Entropy':>8} | {'LR':>9} | {'Time':>7}")
print("-" * 64)

for ep in range(1, N_EPISODES + 1):
    t0    = time.time()
    env   = BangaloreSumoEnv("sumo_config/sim_bangalore.sumocfg")
    state = env.reset()
    total = 0.0
    done  = False

    try:
        while not done:
            action          = agent.choose_action(state)
            next_s, r, done = env.step(action)
            agent.store_reward(r, done)
            agent.update()
            state  = next_s
            total += r

        # Flush any remaining steps in the partial rollout buffer.
        if len(agent.buf_rewards) > 32:
            agent.update()

        agent.lr_decay(ep, N_EPISODES)
        train_rewards.append(total)

        if total > best_reward:
            best_reward = total
            torch.save(agent.net.state_dict(), CHECKPOINT)
            note = "  *"
        else:
            note = ""

        if ep == N_EPISODES:
            train_metrics_last = env.get_all_metrics()

        st      = agent.stats()
        elapsed = time.time() - t0
        print(f"{ep:>4} | {total:>14.0f} | {st['updates']:>7} | "
              f"{st['entropy']:>8.4f} | {st['lr']:>9.6f} | "
              f"{elapsed:>6.0f}s{note}")

    except Exception as exc:
        print(f"{ep:>4} | Episode failed: {exc}")
        env._safe_close()
        time.sleep(1)

    finally:
        env.close()

print(f"\nPhase 1 complete.")
print(f"  Best reward   : {best_reward:.0f}")
print(f"  Last 10 avg   : {np.mean(train_rewards[-10:]):.0f}")
print(f"  First 10 avg  : {np.mean(train_rewards[:10]):.0f}")
print(f"  Improvement   : {np.mean(train_rewards[-10:]) - np.mean(train_rewards[:10]):+.0f}")
print(f"  Total updates : {agent.update_count}")
print(f"\n* = new best reward, checkpoint saved to {CHECKPOINT}")

Phase 1 training — 100 episodes
  Ep |         Reward | Updates |  Entropy |        LR |    Time
----------------------------------------------------------------
   1 |       -2069980 |       0 |   0.0000 |  0.000297 |     58s  *
   2 |       -2004570 |       1 |   0.6929 |  0.000295 |     59s  *
   3 |       -2002540 |       2 |   0.6923 |  0.000292 |     58s  *
   4 |       -2057300 |       3 |   0.6915 |  0.000289 |     58s
   5 |       -2078500 |       4 |   0.6899 |  0.000286 |     58s
   6 |       -1950760 |       5 |   0.6890 |  0.000284 |     58s  *
   7 |       -2071660 |       6 |   0.6912 |  0.000281 |     58s
   8 |       -2047270 |       7 |   0.6915 |  0.000278 |     60s
   9 |       -2056370 |       8 |   0.6892 |  0.000276 |     58s
  10 |       -2111690 |       9 |   0.6898 |  0.000273 |     58s
  11 |       -2073330 |      10 |   0.6902 |  0.000270 |     60s
  12 |       -2050100 |      11 |   0.6926 |  0.000268 |     57s
  13 |       -2150060 |      12 |   0.6822 |  

# Phase 1 Training Visualization

In [11]:
"""
Cell 11: Analyse and visualise Phase 1 training progress.

Three plots are produced:
    1. Episode reward with moving average and first/last-10 reference lines.
    2. Policy loss per update (should trend toward zero as training stabilises).
    3. Policy entropy per update (should decrease as the policy converges).
"""

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.use("Agg")

rewards_arr = np.array(train_rewards)
first_10    = float(np.mean(rewards_arr[:10]))
mid_10      = float(np.mean(rewards_arr[45:55]))
last_10     = float(np.mean(rewards_arr[-10:]))

print("Training reward progression:")
print(f"  Episodes   1-10  mean : {first_10:>12.0f}")
print(f"  Episodes  46-55  mean : {mid_10:>12.0f}")
print(f"  Episodes  91-100 mean : {last_10:>12.0f}")
print(f"  Absolute improvement  : {last_10 - first_10:>+12.0f}")
print(f"  Relative improvement  : {(last_10 - first_10) / abs(first_10) * 100:>+10.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 4), facecolor="#1a1a2e")
fig.suptitle("PPO Phase 1 Training Diagnostics", color="white", fontsize=12)

# --- Reward curve ---
ax = axes[0]
ax.set_facecolor("#0d0d1a")
ax.plot(rewards_arr, color="#a29bfe", alpha=0.4, lw=1)
window = 10
smoothed = np.convolve(rewards_arr, np.ones(window) / window, mode="valid")
ax.plot(range(window - 1, len(rewards_arr)), smoothed,
        color="#a29bfe", lw=2.5, label=f"{window}-ep moving avg")
ax.axhline(first_10, color="#e74c3c", lw=1.0, linestyle="--",
           label=f"First 10 mean: {first_10:.0f}")
ax.axhline(last_10,  color="#2ecc71", lw=1.0, linestyle="--",
           label=f"Last 10 mean:  {last_10:.0f}")
ax.set_title("Episode Reward", color="white", fontsize=10)
ax.set_xlabel("Episode", color="#888")
ax.set_ylabel("Cumulative Reward", color="#888")
ax.tick_params(colors="#888")
ax.legend(fontsize=7, facecolor="#0d0d1a", labelcolor="white", edgecolor="#333")
for sp in ax.spines.values():
    sp.set_edgecolor("#333355")

# --- Policy loss ---
ax2 = axes[1]
ax2.set_facecolor("#0d0d1a")
if agent.policy_losses:
    ax2.plot(agent.policy_losses, color="#e74c3c", lw=1.5)
    ax2.axhline(0, color="white", lw=0.5, linestyle="--", alpha=0.4)
ax2.set_title("Policy Loss  (converges toward zero)", color="white", fontsize=10)
ax2.set_xlabel("Update", color="#888")
ax2.set_ylabel("Loss", color="#888")
ax2.tick_params(colors="#888")
for sp in ax2.spines.values():
    sp.set_edgecolor("#333355")

# --- Entropy ---
ax3 = axes[2]
ax3.set_facecolor("#0d0d1a")
if agent.entropy_hist:
    ax3.plot(agent.entropy_hist, color="#f39c12", lw=1.5)
ax3.set_title("Policy Entropy  (decreases as policy converges)",
              color="white", fontsize=10)
ax3.set_xlabel("Update", color="#888")
ax3.set_ylabel("Entropy", color="#888")
ax3.tick_params(colors="#888")
for sp in ax3.spines.values():
    sp.set_edgecolor("#333355")

plt.tight_layout()
fig.canvas.draw()
plt.savefig("fig_training_phase1.png", dpi=150,
            bbox_inches="tight", facecolor="#1a1a2e")
plt.close()
print("\nSaved: fig_training_phase1.png")

Training reward progression:
  Episodes   1-10  mean :     -2045064
  Episodes  46-55  mean :     -2090075
  Episodes  91-100 mean :     -2052235
  Absolute improvement  :        -7171
  Relative improvement  :       -0.4%

Saved: fig_training_phase1.png


# Phase 2 Training
50 more episodes

In [12]:
"""
Cell 12: Phase 2 fine-tuning — continue training from the best Phase 1
checkpoint for 50 additional episodes at a reduced learning rate.

The lower learning rate (5e-5) allows the policy to refine its behaviour
without destabilising the weights already learned in Phase 1.
"""

import time

import numpy as np
import torch

FINETUNE_LR    = 5e-5
N_FINETUNE     = 50
CHECKPOINT     = "best_ppo_bangalore.pt"

agent.net.load_state_dict(
    torch.load(CHECKPOINT, map_location=DEVICE)
)
for group in agent.opt.param_groups:
    group["lr"] = FINETUNE_LR
agent.lr_now = FINETUNE_LR

print(f"Phase 2 fine-tuning — {N_FINETUNE} episodes")
print(f"  Loaded checkpoint : {CHECKPOINT}")
print(f"  Best reward so far: {best_reward:.0f}")
print(f"  Learning rate     : {FINETUNE_LR}")
print()
print(f"{'Ep':>4} | {'Reward':>14} | {'Entropy':>8} | {'Time':>7}")
print("-" * 42)

for ep in range(1, N_FINETUNE + 1):
    t0    = time.time()
    env   = BangaloreSumoEnv("sumo_config/sim_bangalore.sumocfg")
    state = env.reset()
    total = 0.0
    done  = False

    try:
        while not done:
            action          = agent.choose_action(state)
            next_s, r, done = env.step(action)
            agent.store_reward(r, done)
            agent.update()
            state  = next_s
            total += r

        if len(agent.buf_rewards) > 32:
            agent.update()

        train_rewards.append(total)

        if total > best_reward:
            best_reward = total
            torch.save(agent.net.state_dict(), CHECKPOINT)
            note = "  *"
        else:
            note = ""

        if ep == N_FINETUNE:
            train_metrics_last = env.get_metrics()

        st      = agent.stats()
        elapsed = time.time() - t0
        print(f"{100 + ep:>4} | {total:>14.0f} | "
              f"{st['entropy']:>8.4f} | {elapsed:>6.0f}s{note}")

    except Exception as exc:
        print(f"{100 + ep:>4} | Episode failed: {exc}")
        env._safe_close()
        time.sleep(1)

    finally:
        env.close()

print(f"\nPhase 2 complete.  Total episodes trained: {len(train_rewards)}")
print(f"  Best reward ever : {best_reward:.0f}")
print(f"  Last 10 avg      : {np.mean(train_rewards[-10:]):.0f}")
print(f"\n* = new best reward, checkpoint saved to {CHECKPOINT}")

Phase 2 fine-tuning — 50 episodes
  Loaded checkpoint : best_ppo_bangalore.pt
  Best reward so far: -1937120
  Learning rate     : 5e-05

  Ep |         Reward |  Entropy |    Time
------------------------------------------
 101 |       -2139690 |   0.6898 |     58s
 102 |       -2127530 |   0.6907 |     57s
 103 |       -2171650 |   0.6891 |     59s
 104 |       -2097610 |   0.6907 |     58s
 105 |       -2019330 |   0.6909 |     59s
 106 |       -2070790 |   0.6909 |     58s
 107 |       -2094740 |   0.6902 |     59s
 108 |       -2254180 |   0.6906 |     58s
 109 |       -2063890 |   0.6899 |     59s
 110 |       -2041430 |   0.6884 |     57s
 111 |       -2180620 |   0.6887 |     59s
 112 |       -2059250 |   0.6899 |     57s
 113 |       -2205640 |   0.6896 |     59s
 114 |       -2008430 |   0.6897 |     58s
 115 |       -2130620 |   0.6887 |     58s
 116 |       -2026720 |   0.6866 |     58s
 117 |       -2091460 |   0.6857 |     59s
 118 |       -2056930 |   0.6895 |     58s
 1

# Evaluation of RL Agent vs Fixed-Policy

In [13]:
"""
Cell 13: Load the best model checkpoint and evaluate both the PPO agent
and the Fixed-Cycle baseline.

Each policy is run for N_EVAL_RUNS independent episodes. Per-step metric
arrays are averaged across runs (aligned to the shortest run length to
handle any episode-length variance).
"""

import numpy as np
import torch

N_EVAL_RUNS = 3
CHECKPOINT  = "best_ppo_bangalore.pt"

ALL_METRIC_KEYS = [
    "system_total_stopped",
    "system_total_waiting_time",
    "system_mean_waiting_time",
    "system_mean_speed",
    "avg_waiting_time",
    "avg_travel_time",
    "queue_length",
    "throughput",
    "delay",
]


def run_evaluation(policy_fn, cfg_path: str,
                   label: str, n_runs: int = N_EVAL_RUNS) -> dict:
    """
    Evaluate a policy over multiple independent episodes.

    Parameters
    ----------
    policy_fn : Callable that accepts a state array and returns an action int,
                or an object with a `choose_action(state)` method.
    cfg_path  : Path to the .sumocfg file.
    label     : Display label used in progress output.
    n_runs    : Number of evaluation episodes to average over.

    Returns
    -------
    dict mapping metric key -> np.ndarray of per-step values averaged
    across all runs.
    """
    buffers = {k: [] for k in ALL_METRIC_KEYS}

    for run in range(1, n_runs + 1):
        env   = BangaloreSumoEnv(cfg_path)
        state = env.reset()
        if hasattr(policy_fn, "reset"):
            policy_fn.reset()
        done = False

        try:
            while not done:
                action = (
                    policy_fn(state)
                    if callable(policy_fn)
                    else policy_fn.choose_action(state)
                )
                state, _, done = env.step(action)

            m = env.get_metrics()
            for k in ALL_METRIC_KEYS:
                if k in m:
                    buffers[k].append(m[k])
            print(f"  [{label}] run {run}/{n_runs} complete")

        except Exception as exc:
            print(f"  [{label}] run {run} failed: {exc}")

        finally:
            env.close()

    averaged = {}
    for k, runs in buffers.items():
        if not runs:
            continue
        min_len      = min(len(r) for r in runs)
        averaged[k]  = np.mean([r[:min_len] for r in runs], axis=0)
    return averaged


# Load the best weights before evaluation.
agent.net.load_state_dict(
    torch.load(CHECKPOINT, map_location=DEVICE)
)
agent.net.eval()
print(f"Loaded checkpoint: {CHECKPOINT}\n")

print("Evaluating PPO agent...")
ppo_metrics = run_evaluation(
    lambda s: agent.net.get_action(s)[0],
    "sumo_config/sim_bangalore.sumocfg",
    "PPO",
)

print("\nEvaluating Fixed-Cycle baseline...")
FIXED_POLICY.reset()
fc_metrics = run_evaluation(
    FIXED_POLICY,
    "sumo_config/sim_bangalore.sumocfg",
    "Fixed-Cycle",
)

print("\nEvaluation complete.")

Loaded checkpoint: best_ppo_bangalore.pt

Evaluating PPO agent...
  [PPO] run 1/3 complete
  [PPO] run 2/3 complete
  [PPO] run 3/3 complete

Evaluating Fixed-Cycle baseline...
  [Fixed-Cycle] run 1/3 complete
  [Fixed-Cycle] run 2/3 complete
  [Fixed-Cycle] run 3/3 complete

Evaluation complete.


# Evaluation Metrics

In [15]:
"""
Cell 15: Plot the four benchmark metrics from the 2WSI-RL evaluation
protocol for both training and testing.

Left column  : Per-step metric values from the final training episode.
Right column : Per-step metric values from the evaluation runs,
               comparing PPO against the Fixed-Cycle baseline.
"""

import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

matplotlib.use("Agg")

BENCHMARK_METRICS = [
    ("system_total_stopped",      "Total Stopped Vehicles",  False),
    ("system_total_waiting_time", "Total Waiting Time (s)",  False),
    ("system_mean_waiting_time",  "Mean Waiting Time (s)",   False),
    ("system_mean_speed",         "Mean Speed (m/s)",        True),
]

SMOOTH_WINDOW = 50


def _smooth(arr: np.ndarray, w: int = SMOOTH_WINDOW):
    """Return (smoothed_values, x_indices) via a uniform rolling average."""
    if len(arr) <= w:
        return arr, np.arange(len(arr))
    s  = np.convolve(arr, np.ones(w) / w, mode="valid")
    xi = np.arange(w // 2, w // 2 + len(s))
    return s, xi


fig = plt.figure(figsize=(18, 14), facecolor="#1a1a2e")
fig.suptitle(
    "Bangalore Traffic — Benchmark Metrics  (2WSI-RL evaluation protocol)\n"
    "Left: Training (PPO, last episode)     Right: Testing (PPO vs Fixed-Cycle)",
    color="white", fontsize=13, fontweight="bold", y=0.99,
)
gs = gridspec.GridSpec(
    4, 2, figure=fig,
    hspace=0.50, wspace=0.30,
    left=0.07, right=0.97, top=0.93, bottom=0.05,
)

for row, (key, ylabel, higher_better) in enumerate(BENCHMARK_METRICS):

    # Training column
    ax_tr = fig.add_subplot(gs[row, 0])
    ax_tr.set_facecolor("#0d0d1a")
    if train_metrics_last and key in train_metrics_last:
        arr = train_metrics_last[key]
        ax_tr.plot(arr, color="#a29bfe", alpha=0.18, lw=0.8)
        sm, xi = _smooth(arr)
        ax_tr.plot(xi, sm, color="#a29bfe", lw=2.2,
                   label=f"PPO  (mean={np.mean(arr):.2f})")
        ax_tr.axhline(np.mean(arr), color="#a29bfe",
                      lw=0.8, linestyle=":", alpha=0.6)
    if row == 0:
        ax_tr.set_title("Training  (Bangalore 24 h schedule)",
                        color="#a29bfe", fontsize=10, fontweight="bold")
    ax_tr.set_ylabel(ylabel, color="#aaa", fontsize=8)
    ax_tr.set_xlabel("Simulation step", color="#666", fontsize=7)
    ax_tr.tick_params(colors="#666", labelsize=7)
    ax_tr.legend(fontsize=7, facecolor="#0d0d1a",
                 labelcolor="white", edgecolor="#333")
    for sp in ax_tr.spines.values():
        sp.set_edgecolor("#333355")

    # Testing column
    ax_te = fig.add_subplot(gs[row, 1])
    ax_te.set_facecolor("#0d0d1a")

    p_arr  = ppo_metrics.get(key, np.array([]))
    fc_arr = fc_metrics.get(key,  np.array([]))

    for arr, colour, label, linestyle in [
        (p_arr,  "#2ecc71", "PPO",         "-"),
        (fc_arr, "#e74c3c", "Fixed-Cycle", "--"),
    ]:
        if len(arr):
            ax_te.plot(arr, color=colour, alpha=0.18, lw=0.8)
            sm, xi = _smooth(arr)
            ax_te.plot(xi, sm, color=colour, lw=2.2, linestyle=linestyle,
                       label=f"{label}  (mean={np.mean(arr):.2f})")
            ax_te.axhline(np.mean(arr), color=colour,
                          lw=0.8, linestyle=":", alpha=0.6)

    if len(p_arr) and len(fc_arr):
        pm, fcm = np.mean(p_arr), np.mean(fc_arr)
        ppo_wins = (pm > fcm) if higher_better else (pm < fcm)
        badge    = "PPO"         if ppo_wins else "Fixed-Cycle"
        b_colour = "#2ecc71"     if ppo_wins else "#e74c3c"
        ax_te.text(
            0.98, 0.95, f"{badge} lower" if not higher_better
            else f"{badge} faster",
            transform=ax_te.transAxes, ha="right", va="top",
            color=b_colour, fontsize=8, fontweight="bold",
            bbox=dict(facecolor="#0d0d1a", edgecolor=b_colour,
                      boxstyle="round,pad=0.3", alpha=0.85),
        )

    if row == 0:
        ax_te.set_title("Testing — PPO vs Fixed-Cycle",
                        color="#fdcb6e", fontsize=10, fontweight="bold")
    ax_te.set_ylabel(ylabel, color="#aaa", fontsize=8)
    ax_te.set_xlabel("Simulation step", color="#666", fontsize=7)
    ax_te.tick_params(colors="#666", labelsize=7)
    ax_te.legend(fontsize=7, facecolor="#0d0d1a",
                 labelcolor="white", edgecolor="#333")
    for sp in ax_te.spines.values():
        sp.set_edgecolor("#333355")

fig.canvas.draw()
plt.savefig("fig_benchmark_metrics.png", dpi=150,
            bbox_inches="tight", facecolor="#1a1a2e")
plt.close()
print("Saved: fig_benchmark_metrics.png")

Saved: fig_benchmark_metrics.png


In [16]:
"""
Cell 16: Plot the five additional performance metrics, each with two panels:
    Left  : Full 24-hour simulation curve, with traffic period shading.
    Right : Per-period bar chart comparing PPO and Fixed-Cycle.

Traffic period shading colours:
    Blue   = Night
    Red    = Rush hour (morning and evening)
    Orange = Shoulder periods (mid-morning, evening wind-down)
    Grey   = Afternoon
"""

import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

matplotlib.use("Agg")

ADDITIONAL_METRICS = [
    ("avg_waiting_time", "Average Waiting Time (s / vehicle)", False),
    ("avg_travel_time",  "Average Travel Time (s / vehicle)",  False),
    ("queue_length",     "Queue Length (vehicles)",             False),
    ("throughput",       "Throughput (vehicles / step)",        True),
    ("delay",            "Delay vs Free-Flow (s / vehicle)",    False),
]

SECONDS_PER_HOUR = 3600
STEP_LENGTH      = 10

# Period boundaries in simulation steps
PERIODS = [
    (0,                            8  * SECONDS_PER_HOUR // STEP_LENGTH,
     "#3498db", "Night"),
    (8  * SECONDS_PER_HOUR // STEP_LENGTH,
     11 * SECONDS_PER_HOUR // STEP_LENGTH, "#e74c3c", "Morning rush"),
    (11 * SECONDS_PER_HOUR // STEP_LENGTH,
     13 * SECONDS_PER_HOUR // STEP_LENGTH, "#f39c12", "Mid-morning"),
    (13 * SECONDS_PER_HOUR // STEP_LENGTH,
     16 * SECONDS_PER_HOUR // STEP_LENGTH, "#95a5a6", "Afternoon"),
    (16 * SECONDS_PER_HOUR // STEP_LENGTH,
     20 * SECONDS_PER_HOUR // STEP_LENGTH, "#e74c3c", "Evening rush"),
    (20 * SECONDS_PER_HOUR // STEP_LENGTH,
     23 * SECONDS_PER_HOUR // STEP_LENGTH, "#f39c12", "Evening"),
    (23 * SECONDS_PER_HOUR // STEP_LENGTH,
     86400 // STEP_LENGTH,                  "#3498db", "Night"),
]

PERIOD_LABELS = ["Night", "Morn rush", "Mid-morn",
                 "Afternoon", "Eve rush", "Evening"]

fig = plt.figure(figsize=(20, 18), facecolor="#1a1a2e")
fig.suptitle(
    "Bangalore Traffic — Additional Performance Metrics\n"
    "Left: Full 24 h simulation curve     Right: Breakdown by traffic period",
    color="white", fontsize=13, fontweight="bold", y=0.99,
)
gs = gridspec.GridSpec(
    5, 2, figure=fig,
    hspace=0.55, wspace=0.30,
    left=0.07, right=0.97, top=0.95, bottom=0.04,
)

for row, (key, ylabel, higher_better) in enumerate(ADDITIONAL_METRICS):
    p_arr  = ppo_metrics.get(key, np.array([]))
    fc_arr = fc_metrics.get(key, np.array([]))
    max_len = max(len(p_arr), len(fc_arr), 1)

    # Left panel: full simulation curve with period shading
    ax_l = fig.add_subplot(gs[row, 0])
    ax_l.set_facecolor("#0d0d1a")

    for start, end, colour, _ in PERIODS:
        ax_l.axvspan(start, min(end, max_len), alpha=0.07, color=colour)

    for arr, colour, label, linestyle in [
        (p_arr,  "#2ecc71", "PPO",         "-"),
        (fc_arr, "#e74c3c", "Fixed-Cycle", "--"),
    ]:
        if len(arr):
            ax_l.plot(arr, color=colour, alpha=0.20, lw=0.8)
            sm, xi = _smooth(arr)
            ax_l.plot(xi, sm, color=colour, lw=2.0, linestyle=linestyle,
                      label=f"{label}  (mean={np.mean(arr):.2f})")

    ax_l.set_title(ylabel, color="white", fontsize=10, fontweight="bold")
    ax_l.set_ylabel(ylabel.split(" (")[0], color="#aaa", fontsize=8)
    ax_l.set_xlabel("Simulation step", color="#666", fontsize=7)
    ax_l.tick_params(colors="#666", labelsize=7)
    ax_l.legend(fontsize=7, facecolor="#0d0d1a",
                labelcolor="white", edgecolor="#333")
    for sp in ax_l.spines.values():
        sp.set_edgecolor("#333355")

    # Right panel: per-period bar chart
    ax_r = fig.add_subplot(gs[row, 1])
    ax_r.set_facecolor("#0d0d1a")

    p_by_period:  list = []
    fc_by_period: list = []

    for start, end, _, _ in PERIODS[:-1]:
        for arr, target in [(p_arr, p_by_period), (fc_arr, fc_by_period)]:
            s = min(start, len(arr))
            e = min(end,   len(arr))
            target.append(float(np.mean(arr[s:e])) if e > s else 0.0)

    x    = np.arange(len(PERIOD_LABELS))
    w    = 0.35
    cols = [p[2] for p in PERIODS[:-1]]

    if p_by_period:
        ax_r.bar(x - w / 2, p_by_period, w, color=cols, alpha=0.85,
                 edgecolor="white", lw=0.5, label="PPO")
    if fc_by_period:
        ax_r.bar(x + w / 2, fc_by_period, w, color=cols, alpha=0.45,
                 edgecolor="white", lw=0.5, hatch="//", label="Fixed-Cycle")

    ax_r.set_xticks(x)
    ax_r.set_xticklabels(PERIOD_LABELS, rotation=20,
                         ha="right", color="#aaa", fontsize=7)
    ax_r.set_title("By traffic period", color="white",
                   fontsize=9, fontweight="bold")
    ax_r.tick_params(colors="#666", labelsize=7)
    ax_r.legend(fontsize=7, facecolor="#0d0d1a",
                labelcolor="white", edgecolor="#333")
    for sp in ax_r.spines.values():
        sp.set_edgecolor("#333355")

fig.canvas.draw()
plt.savefig("fig_additional_metrics.png", dpi=150,
            bbox_inches="tight", facecolor="#1a1a2e")
plt.close()
print("Saved: fig_additional_metrics.png")

Saved: fig_additional_metrics.png


# Final Results

In [22]:
"""
Cell 17: Print the complete evaluation results table for all nine metrics.

Differences below the tie threshold are recorded as ties rather than
assigning a spurious winner, which was causing throughput to be awarded
to Fixed-Cycle despite an effectively zero difference.
"""

import numpy as np

ALL_METRICS_DEF = [
    # (metric_key, display_label, higher_is_better, unit)
    # Benchmark metrics  (2WSI-RL evaluation protocol)
    ("system_total_stopped",      "Total Stopped Vehicles",  False, "vehicles"),
    ("system_total_waiting_time", "Total Waiting Time",      False, "seconds"),
    ("system_mean_waiting_time",  "Mean Waiting Time",       False, "s / vehicle"),
    ("system_mean_speed",         "Mean Speed",              True,  "m/s"),
    # Additional metrics
    ("avg_waiting_time",          "Avg Waiting Time",        False, "s / vehicle"),
    ("avg_travel_time",           "Avg Travel Time",         False, "seconds"),
    ("queue_length",              "Queue Length",            False, "vehicles"),
    ("throughput",                "Throughput",              True,  "veh / step"),
    ("delay",                     "Delay vs Free-Flow",      False, "seconds"),
]

# Minimum absolute difference required to declare a winner.
# Values below the threshold are reported as ties.
TIE_THRESHOLD = {
    "system_total_stopped"     : 0.05,
    "system_total_waiting_time": 1.0,
    "system_mean_waiting_time" : 0.5,
    "system_mean_speed"        : 0.005,
    "avg_waiting_time"         : 0.5,
    "avg_travel_time"          : 1.0,
    "queue_length"             : 0.05,
    "throughput"               : 0.01,
    "delay"                    : 0.5,
}

W = 92

def _divider(char: str = "-", width: int = W) -> None:
    print(char * width)


print("\n" + "=" * W)
print("  BANGALORE TRAFFIC SIMULATION — PPO vs Fixed-Cycle")
print("  16,400 vehicles / day   12 turn movements   3 vehicle types")
print("=" * W)
print(f"  {'Metric':<32} {'Unit':<14} {'PPO':>10} "
      f"{'Fixed':>10} {'Delta':>10} {'Outcome':<16}")
_divider()

wins_ppo = wins_fixed = ties = 0
section  = ""

for key, label, higher_better, unit in ALL_METRICS_DEF:
    # Section headers
    if key == "system_total_stopped" and section != "benchmark":
        print("  -- Benchmark metrics (2WSI-RL protocol) " + "-" * 45)
        section = "benchmark"
    if key == "avg_waiting_time" and section != "additional":
        _divider()
        print("  -- Additional metrics " + "-" * 64)
        section = "additional"

    p_arr  = ppo_metrics.get(key, np.array([]))
    fc_arr = fc_metrics.get(key,  np.array([]))

    if not len(p_arr) or not len(fc_arr):
        print(f"  {label:<32} {unit:<14} {'N/A':>10} {'N/A':>10}")
        continue

    pm, fcm   = float(np.mean(p_arr)), float(np.mean(fc_arr))
    delta     = pm - fcm
    threshold = TIE_THRESHOLD.get(key, 0.01)

    if abs(delta) < threshold:
        outcome = "Tie"
        ties   += 1
    elif (delta > 0) if higher_better else (delta < 0):
        outcome = "PPO"
        wins_ppo += 1
    else:
        outcome = "Fixed-Cycle"
        wins_fixed += 1

    print(f"  {label:<32} {unit:<14} {pm:>10.3f} "
          f"{fcm:>10.3f} {delta:>+10.3f} {outcome:<16}")

_divider("=")

verdict = (
    "PPO outperforms Fixed-Cycle" if wins_ppo   > wins_fixed
    else "Fixed-Cycle outperforms PPO" if wins_fixed > wins_ppo
    else "Draw"
)
print(f"\n  Score  —  PPO: {wins_ppo}   Fixed-Cycle: {wins_fixed}   Ties: {ties}")
print("=" * W)


  BANGALORE TRAFFIC SIMULATION — PPO vs Fixed-Cycle
  16,400 vehicles / day   12 turn movements   3 vehicle types
  Metric                           Unit                  PPO      Fixed      Delta Outcome         
--------------------------------------------------------------------------------------------
  -- Benchmark metrics (2WSI-RL protocol) ---------------------------------------------
  Total Stopped Vehicles           vehicles            6.773      6.997     -0.224 PPO             
  Total Waiting Time               seconds           264.106    269.153     -5.047 PPO             
  Mean Waiting Time                s / vehicle        15.691     14.968     +0.723 Fixed-Cycle     
  Mean Speed                       m/s                 3.005      2.959     +0.047 PPO             
--------------------------------------------------------------------------------------------
  -- Additional metrics ----------------------------------------------------------------
  Avg Waiting Time    